# Gold Fact Sales Enriched with Holidays

Enrich sales transactions with:
- Customer information (from dim_customers)
- Product information (from dim_products)
- **Holiday information** (from silver.holidays)
  - Holiday flags (is it a holiday?)
  - Holiday names and types (major holidays like Christmas, Easter)
  - Pre/post holiday week flags for seasonal analysis

**Business Questions This Enables:**
- Do sales spike before major holidays?
- Are certain products more popular during holidays?
- How do holiday sales differ by country?

# The Transformation Logic

In [0]:
query = """
SELECT 
  sd.order_number,
  sd.product_number,
  sd.customer_id,
  sd.order_date,
  sd.ship_date,
  sd.due_date,
  sd.sales_amount,
  sd.quantity,
  sd.price,
  loc.country,
  loc.country_code,
  w.temp_max_c AS temperature_max,
  w.temp_min_c AS temperature_min,
  w.precipitation_mm AS precipitation,
  w.wind_speed_kmh AS wind_speed,
  w.weather_code
FROM silver.crm_sales_cdc sd
LEFT JOIN workspace.silver.erp_customer_location_cdc loc
  ON sd.customer_id = loc.customer_number
LEFT JOIN workspace.silver.weather_daily w
  ON loc.country_code = w.country_code
  AND sd.order_date = w.date
"""

df = spark.sql(query)

In [0]:
query = """
SELECT 
  sd.order_number,
  pr.product_key,
  cu.customer_key,
  sd.order_date,
  sd.ship_date,
  sd.due_date,
  sd.sales_amount,
  sd.quantity,
  sd.price,
  
  -- Holiday information
  h.holiday_date,
  h.holiday_name,
  h.is_major_holiday,
  h.is_global,
  
  -- Holiday proximity flags
  CASE 
    WHEN h.holiday_date IS NOT NULL THEN TRUE 
    ELSE FALSE 
  END AS is_holiday,
  
  CASE
    WHEN h.holiday_date IS NOT NULL AND DATEDIFF(h.holiday_date, sd.order_date) BETWEEN 0 AND 7 THEN TRUE
    ELSE FALSE
  END AS is_pre_holiday_week,
  
  CASE
    WHEN h.holiday_date IS NOT NULL AND DATEDIFF(sd.order_date, h.holiday_date) BETWEEN 0 AND 7 THEN TRUE
    ELSE FALSE
  END AS is_post_holiday_week
  
FROM silver.crm_sales_cdc sd 
LEFT JOIN gold.dim_products pr 
  ON sd.product_number = pr.product_number
  AND pr.is_current = TRUE
LEFT JOIN gold.dim_customers cu   
  ON sd.customer_id = cu.customer_id
  AND cu.is_current = TRUE
LEFT JOIN silver.holidays_cdc h
  ON sd.order_date = h.holiday_date
  AND h.country_code = CASE 
    WHEN cu.country = 'United States' THEN 'US'
    WHEN cu.country = 'United Kingdom' THEN 'UK'
    WHEN cu.country = 'Germany' THEN 'DE'
    WHEN cu.country = 'France' THEN 'FR'
    WHEN cu.country = 'Canada' THEN 'CA'
    WHEN cu.country = 'Australia' THEN 'AU'
    ELSE NULL
  END
"""

df = spark.sql(query)

In [0]:
df.limit(10).display()

# Writing Gold Table

In [0]:
# Write to gold layer
df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("workspace.gold.fact_sales_enriched")

print("✓ Gold table created: workspace.gold.fact_sales_enriched")

## Sanity checks of gold table

In [0]:
%sql
SELECT *
FROM workspace.gold.fact_sales_enriched
WHERE holiday_date IS NOT NULL
LIMIT 10;

Analyse holiday sales by country

In [0]:
%sql
SELECT 
  cu.country,
  COUNT(*) as total_orders,
  SUM(CASE WHEN fe.is_holiday THEN 1 ELSE 0 END) as holiday_orders,
  SUM(CASE WHEN fe.is_major_holiday THEN 1 ELSE 0 END) as major_holiday_orders,
  SUM(fe.sales_amount) as total_sales,
  SUM(CASE WHEN fe.is_holiday THEN fe.sales_amount ELSE 0 END) as holiday_sales,
  ROUND(100.0 * SUM(CASE WHEN fe.is_holiday THEN 1 ELSE 0 END) / COUNT(*), 2) as holiday_order_pct,
  ROUND(100.0 * SUM(CASE WHEN fe.is_holiday THEN fe.sales_amount ELSE 0 END) / SUM(fe.sales_amount), 2) as holiday_sales_pct
FROM workspace.gold.fact_sales_enriched fe
LEFT JOIN gold.dim_customers cu
  ON fe.customer_key = cu.customer_key
WHERE cu.country IS NOT NULL AND cu.country != 'n/a'
GROUP BY cu.country
ORDER BY holiday_sales DESC